# Evaluation & comparison

Reads saved results only, no training here (embed-and-pool training lives in `embedding.ipynb`).

1. Overview of all systems (Project 1 + Project 2)
2. Learning curves: overfitting
3. Bayesian comparison of fold scores (with a ROPE)
4. Comment-level evaluation: macro-F1, per-emotion F1, confusions, McNemar
5. Error analysis: what makes a comment hard, and where does the LSTM differ from embed-and-pool?

**What this notebook needs from the training notebooks** (helpers in `projects/src/evaluation.py`):
- `results/<name>.json`, written by `save_results(name, out, ...)` → sections 1–3
- `results/predictions/<name>.npz`, written by `evaluation.save_predictions(name, out, folds)` right after `cross_validate` → sections 4–5

Both only work if the system was trained on Project 1's folds (`p1_experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)`).

In [ ]:
# Setup: make the course code (part1, part2) and our shared code (projects/src) importable.
# Run from projects/project2 (VS Code / JupyterLab do this by default).
import sys
from pathlib import Path

PROJECT2_DIR = Path.cwd().resolve()
REPO_ROOT = PROJECT2_DIR.parents[1]
PROJECTS_DIR = REPO_ROOT / "projects"
for p in (REPO_ROOT, PROJECTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from part1 import analysis
from part1 import experiment as p1_experiment
from src import config, data, evaluation

In [ ]:
# System names (= file names in results/). Rename LSTM to whatever name the LSTM results are saved under.
BEST_P1 = "mlp__indicator"         # best Project 1 system (55.2 %)
POOL = "mlp__glove200-pool"        # embed-and-pool, frozen GloVe Twitter 200d
LSTM = "lstm__glove200"            # <- TODO: match the LSTM results name

ROPE = 0.01   # same as Project 1: 1 accuracy point, about the fold-to-fold std

## 1 · Overview

In [ ]:
records = evaluation.load_records()
scores = evaluation.fold_scores(records)
evaluation.overview(records)

In [ ]:
analysis.plot_box_and_points(evaluation.data_frame(records), metric="accuracy")

## 2 · Learning curves: overfitting

Mean over the folds (band = min–max over folds); the dot marks the epoch with the best validation accuracy.
The CV scores use the epoch with the lowest validation loss.

In [ ]:
p2_names = [n for n, r in records.items() if r["project"] == "P2"]
fig = evaluation.plot_learning_curves(records, p2_names)
evaluation.save(fig, "learning_curves")
plt.show()

fig = evaluation.plot_learning_curves(records, p2_names, metric="loss")
plt.show()

evaluation.overfitting_table(records, p2_names)

**Conclusion.** _TODO_

## 3 · Bayesian comparison of fold accuracies

Benavoli correlated t-test (course code), A vs. B, mean difference = B − A.
All systems use the same 5 folds, so the comparison is paired fold by fold.

In [ ]:
comparisons = [
    (BEST_P1, POOL),   # did pretrained embeddings help over last week's best?
    (BEST_P1, LSTM),   # did the recurrent encoder?
    (POOL, LSTM),      # does the LSTM improve meaningfully on embed-and-pool?
]
evaluation.bayesian_comparisons(scores, comparisons, rope=ROPE)

**Conclusion.** _TODO_

## 4 · Comment-level evaluation

GoEmotions (fine) has 28 very imbalanced classes (`neutral` alone is about a third of the data), so accuracy hides how
the rare emotions do. Macro-F1 weights every emotion equally.

Every comment is in exactly one validation fold, so each system has one out-of-fold prediction per comment.

In [ ]:
LABEL_MODE = "fine"
pairs, outcome_space, _ = data.load_pairs(label_mode=LABEL_MODE)
folds = p1_experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)
labels = np.array([y for _, y in pairs])
class_names = [outcome_space.name(i) for i in range(len(outcome_space))]

preds, pred_folds = {}, {}
for name in evaluation.systems_with_predictions():
    preds[name], pred_folds[name] = evaluation.load_predictions(name, len(pairs))
correct = {n: p == labels for n, p in preds.items()}

if not preds:
    raise RuntimeError("No saved predictions yet: run evaluation.save_predictions(name, out, folds) after cross_validate.")
print("systems with predictions:", list(preds))

In [ ]:
evaluation.metrics_table(labels, preds, pred_folds)

In [ ]:
# Same Bayesian test, but on macro-F1 per fold
f1_scores = evaluation.macro_f1_per_fold(labels, preds, pred_folds)
evaluation.bayesian_comparisons(f1_scores, [(POOL, LSTM)], rope=ROPE)

In [ ]:
f1_table = evaluation.per_class_f1(labels, preds, class_names)
fig = evaluation.plot_per_class_f1(f1_table)
evaluation.save(fig, "per_class_f1")
plt.show()
f1_table.style.format("{:.3f}", subset=list(preds)).background_gradient(cmap="Blues", subset=list(preds), vmin=0, vmax=1)

In [ ]:
for name, p in preds.items():
    fig = evaluation.plot_confusion(labels, p, class_names, title=name)
    evaluation.save(fig, f"confusion__{name}")
    plt.show()
    display(evaluation.top_confusions(labels, p, class_names))

**Paired test per comment (McNemar).** Of the comments where exactly one system is right, is it evenly split?
This uses all 45k comments, so even tiny differences become significant: read it together with the effect size
(acc A vs. acc B) and the ROPE result above.

In [ ]:
evaluation.mcnemar_table(labels, preds, [(POOL, LSTM)])

**Conclusion.** _TODO_

## 5 · Error analysis

Features per comment that might explain errors. Same tokenizer as the encoders (`BasicTokenizer`, lowercase, keep punctuation).
OOV is measured against the GloVe Twitter vocabulary; rare = seen fewer than 5 times in the training part of the comment's own fold.

In [ ]:
import nltk
from part2 import clslib

nltk.download("punkt_tab", quiet=True)
tokenizer = clslib.BasicTokenizer(lowercase=True, remove_punct=False, remove_stopwords=False)

# all GloVe Twitter sizes share one vocabulary, so read the words from the small 25d file
glove_txt = PROJECT2_DIR / "data" / "cache" / "glove.twitter.27B.25d.txt"
glove_vocab = evaluation.glove_vocabulary(glove_txt)

features = evaluation.text_features(pairs, folds, tokenizer, glove_vocab)
features.describe().T

In [ ]:
for feature, bins in [("n_tokens", [0, 5, 10, 15, 20, 30, 200]),
                      ("oov_rate", [0, 0.001, 0.05, 0.1, 0.2, 1]),
                      ("rare_rate", [0, 0.001, 0.05, 0.1, 0.2, 1])]:
    table = evaluation.accuracy_by_bins(features, correct, feature, bins)
    fig = evaluation.plot_accuracy_by_bins(table, feature)
    evaluation.save(fig, f"accuracy_by_{feature}")
    plt.show()
    display(table)

# binary features: accuracy with (1) / without (0)
pd.DataFrame({f: {f"{n} ({v})": c[features[f] == v].mean() for n, c in correct.items() for v in (0, 1)}
              for f in ["negation", "has_name", "has_emoji"]}).T

**Which features explain correctness?** L1 logistic regression per system (features standardised).
Weight < 0: the system struggles as the feature grows; ≈ 0: irrelevant.

In [ ]:
weights = evaluation.l1_correctness_weights(features, correct)
weights.style.format("{:+.3f}").background_gradient(cmap="RdBu", vmin=-0.5, vmax=0.5)

**Bayesian version (course BDA model).** Same question with posterior uncertainty, on a random subsample
(the same comments for every system). Takes a few minutes, so it is off by default.

In [ ]:
RUN_BDA = False

if RUN_BDA:
    import arviz as az
    import part2.analysis

    names, sys_id, x, c = evaluation.bda_inputs(features, correct, sample=3000)
    mcmc = part2.analysis.fit_bda_model(num_systems=len(names), sys_id=sys_id, features=x,
                                        correctness=c, num_chains=2)
    posterior = part2.analysis.posterior_to_arviz(mcmc, system_names=names, feature_names=list(features.columns))
    az.plot_forest(posterior, var_names=["accuracy"], combined=True)
    plt.show()
    az.plot_forest(posterior, var_names=["w"], combined=True, figure_kwargs={"figsize": (8, 8)})
    plt.show()

**Where do the systems disagree?** Comments only the LSTM gets right vs. only embed-and-pool gets right.
If word order matters, we expect negation and longer comments among the LSTM's wins.

In [ ]:
if POOL in correct and LSTM in correct:
    groups = {"only LSTM right": correct[LSTM] & ~correct[POOL],
              "only pool right": ~correct[LSTM] & correct[POOL],
              "both right": correct[LSTM] & correct[POOL],
              "both wrong": ~correct[LSTM] & ~correct[POOL]}
    profile = pd.DataFrame({g: features[m].mean() for g, m in groups.items()}).T
    profile.insert(0, "comments", [m.sum() for m in groups.values()])
    display(profile)

    show = pd.DataFrame({"text": [t for t, _ in pairs], "gold": [class_names[y] for y in labels],
                         "pool": [class_names[y] for y in preds[POOL]],
                         "lstm": [class_names[y] for y in preds[LSTM]]})
    pd.set_option("display.max_colwidth", 200)
    display(show[groups["only LSTM right"]].sample(10, random_state=config.SEED))
    display(show[groups["only pool right"]].sample(10, random_state=config.SEED))
else:
    print("Needs saved predictions for both", POOL, "and", LSTM)

**Conclusion.** _TODO_